# 02 - Training (Simple)

Bu notebook preprocess'ten gelen `.npz` dosyasını yükler ve **basit** şekilde model eğitir.

Desteklenen seçenekler:
- Logistic Regression (from scratch, NumPy ile hızlı)
- (Opsiyonel) Logistic Regression (PyTorch ile CUDA varsa GPU)
- (Opsiyonel) Random Forest (scikit-learn ile hızlı baseline)

> Eğer kütüphane kullanımı kuralı seni zorluyorsa: `MODEL_TYPE = "logreg_numpy"` kısmı en sade ve hızlı seçenektir (ML framework kullanmıyor).

In [ ]:

import os
import json
from pathlib import Path
import numpy as np

# Optional (only if you want)
USE_TORCH = False
USE_SKLEARN_RF = False

# If you want to try CUDA:
# USE_TORCH = True

# Reproducibility
SEED = 42
np.random.seed(SEED)

PREPROCESSED_DIR = Path("./preprocessed")
MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("PREPROCESSED_DIR:", PREPROCESSED_DIR.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())


## 1) Dataset seçimi

Aşağıdakilerden birini seç:
- `plantvillage_preprocessed.npz`
- `plant_disease_detection_preprocessed.npz`
- `plantdoc_converted_preprocessed.npz`

In [ ]:

DATASET_FILE = PREPROCESSED_DIR / "plant_disease_detection_preprocessed.npz"  # change
OUT_PREFIX = DATASET_FILE.stem.replace("_preprocessed","")  # used in saved model names

data = np.load(DATASET_FILE, allow_pickle=True)

X_train = data["X_train"].astype(np.float32)
y_train = data["y_train"].astype(np.int64)
X_val   = data["X_val"].astype(np.float32)
y_val   = data["y_val"].astype(np.int64)

class_names = list(data["class_names"])
num_classes = len(class_names)
num_features = X_train.shape[1]

print("Loaded:", DATASET_FILE)
print("train:", X_train.shape, "val:", X_val.shape, "classes:", num_classes, "features:", num_features)


## 2) Basit metrik fonksiyonları

In [ ]:

def accuracy(y_true, y_pred):
    return float((y_true == y_pred).mean())

def macro_precision_recall_f1(y_true, y_pred, num_classes):
    # Simple macro average (no sklearn)
    eps = 1e-12
    precisions, recalls, f1s = [], [], []
    for c in range(num_classes):
        tp = int(((y_true == c) & (y_pred == c)).sum())
        fp = int(((y_true != c) & (y_pred == c)).sum())
        fn = int(((y_true == c) & (y_pred != c)).sum())

        p = tp / (tp + fp + eps)
        r = tp / (tp + fn + eps)
        f1 = 2 * p * r / (p + r + eps)

        precisions.append(p); recalls.append(r); f1s.append(f1)

    return float(np.mean(precisions)), float(np.mean(recalls)), float(np.mean(f1s))


## 3) Logistic Regression (NumPy, hızlı)

- Mini-batch gradient descent
- Early stopping (val accuracy)
- L2 regularization

Bu kısım genelde **en hızlı + en stabil** sonuç verir.

In [ ]:

def softmax_np(logits):
    # logits: (B, C)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / (exp.sum(axis=1, keepdims=True) + 1e-12)

def train_logreg_numpy(
    X_train, y_train, X_val, y_val,
    num_classes,
    lr=0.1,
    epochs=30,
    batch_size=512,
    l2=1e-4,
    patience=5,
):
    n, d = X_train.shape
    W = (0.01 * np.random.randn(d, num_classes)).astype(np.float32)
    b = np.zeros((1, num_classes), dtype=np.float32)

    best_val = -1.0
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        # shuffle
        idx = np.random.permutation(n)
        Xs, ys = X_train[idx], y_train[idx]

        # mini-batches
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            xb = Xs[start:end]
            yb = ys[start:end]

            logits = xb @ W + b  # (B, C)
            probs = softmax_np(logits)

            # one-hot (B,C) without big memory: subtract 1 at correct class
            grad_logits = probs
            grad_logits[np.arange(len(yb)), yb] -= 1.0
            grad_logits /= float(len(yb))

            # gradients
            grad_W = xb.T @ grad_logits + l2 * W
            grad_b = grad_logits.sum(axis=0, keepdims=True)

            # update
            W -= lr * grad_W
            b -= lr * grad_b

        # eval
        val_pred = (X_val @ W + b).argmax(axis=1)
        val_acc = accuracy(y_val, val_pred)

        print(f"Epoch {epoch:02d} | val_acc={val_acc:.4f}")

        if val_acc > best_val + 1e-5:
            best_val = val_acc
            best_state = (W.copy(), b.copy())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break

        # simple lr decay
        lr *= 0.95

    W, b = best_state
    return {"W": W, "b": b, "best_val_acc": best_val}


## 4) (Opsiyonel) Logistic Regression (PyTorch + CUDA)

Bu kısım **cuda** varsa hızlıdır. Yoksa kapalı bırak.

In [ ]:

def train_logreg_torch(X_train, y_train, X_val, y_val, num_classes, lr=0.1, epochs=20, batch_size=1024, patience=5):
    import torch
    import torch.nn as nn

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Torch device:", device)

    Xtr = torch.tensor(X_train, dtype=torch.float32, device=device)
    ytr = torch.tensor(y_train, dtype=torch.long, device=device)
    Xva = torch.tensor(X_val, dtype=torch.float32, device=device)
    yva = torch.tensor(y_val, dtype=torch.long, device=device)

    model = nn.Linear(X_train.shape[1], num_classes).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    best_val = -1.0
    best_state = None
    bad_epochs = 0

    n = X_train.shape[0]
    for epoch in range(1, epochs + 1):
        # shuffle indices
        perm = torch.randperm(n, device=device)

        model.train()
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            idx = perm[start:end]
            xb = Xtr[idx]
            yb = ytr[idx]

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        # eval
        model.eval()
        with torch.no_grad():
            val_pred = torch.argmax(model(Xva), dim=1)
            val_acc = float((val_pred == yva).float().mean().item())

        print(f"Epoch {epoch:02d} | val_acc={val_acc:.4f}")

        if val_acc > best_val + 1e-5:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break

        for g in opt.param_groups:
            g["lr"] *= 0.95

    return {"state_dict": best_state, "best_val_acc": best_val}


## 5) Eğitim + Kaydetme

`MODEL_TYPE` seç:
- `logreg_numpy` (önerilen)
- `logreg_torch` (cuda denemek için)

Eğitim bitince `models/` içine kaydeder.

In [ ]:

MODEL_TYPE = "logreg_numpy"   # "logreg_numpy" or "logreg_torch"

if MODEL_TYPE == "logreg_numpy":
    model = train_logreg_numpy(
        X_train, y_train, X_val, y_val,
        num_classes=num_classes,
        lr=0.2,
        epochs=30,
        batch_size=1024,
        l2=1e-4,
        patience=6,
    )
    out_path = MODEL_DIR / f"{OUT_PREFIX}_logreg_numpy.npz"
    np.savez_compressed(out_path, W=model["W"], b=model["b"], class_names=np.array(class_names, dtype=object))
    print("Saved model:", out_path)

elif MODEL_TYPE == "logreg_torch":
    if not USE_TORCH:
        raise RuntimeError("Set USE_TORCH = True in the first cell to use torch.")
    model = train_logreg_torch(
        X_train, y_train, X_val, y_val,
        num_classes=num_classes,
        lr=0.2,
        epochs=20,
        batch_size=2048,
        patience=6,
    )
    out_path = MODEL_DIR / f"{OUT_PREFIX}_logreg_torch.pt"
    import torch
    payload = {"state_dict": model["state_dict"], "class_names": class_names, "num_features": int(num_features)}
    torch.save(payload, out_path)
    print("Saved model:", out_path)

else:
    raise ValueError("Unknown MODEL_TYPE")
